# 1. Data Cleaning

**Project:** Customer churn prediction for a UK online gift wholesaler (UCI Online Retail II).

**Goal of this notebook:** understand the raw transactions, clean them, and save one table of real customer purchases. The next notebook turns these purchases into customer features.

In [1]:
import pandas as pd

# Data Understanding

In [2]:
df = pd.read_excel(
    "data/online_retail_II.xlsx",
    sheet_name="Year 2009-2010",
    dtype={"Invoice": "string", "StockCode": "string", "Customer ID": "string"}
)

In [3]:
df_next_year = pd.read_excel(
    "data/online_retail_II.xlsx",
    sheet_name="Year 2010-2011",
    dtype={"Invoice": "string", "StockCode": "string", "Customer ID": "string"}
)

In [4]:
df = pd.concat([df, df_next_year], ignore_index=True)
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [5]:
df.shape

(1067371, 8)

Each row is an invoice-product transaction line. Both yearly sheets are combined in one DataFrame with the eight original columns.

# Data Exploration

In [6]:
# Check data types.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  string        
 1   StockCode    1067371 non-null  string        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   string        
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), object(1), str(1), string(3)
memory usage: 94.2+ MB


**Data types to review:** Text columns use both `object` and `string`; different types alone do not mean errors. We will decide conversion/encoding after choosing model inputs. IDs and dates may still be needed to build purchase histories.

In [7]:
# Summary statistics.
df[["Quantity", "Price"]].describe().round(2)

,Quantity,Price
count,1067371.00,1067371.00
mean,9.94,4.65
std,172.71,123.55
min,-80995.00,-53594.36
25%,1.00,1.25
50%,3.00,2.10
75%,10.00,4.15
max,80995.00,38970.00


In [8]:
# Distinct values before cleaning.
df.nunique()

Invoice        53628
StockCode       5305
Description     5698
Quantity        1057
InvoiceDate    47635
Price           2807
Customer ID     5942
Country           43
dtype: int64

In [9]:
# Date range.
df["InvoiceDate"].agg(["min", "max"])

min   2009-12-01 07:45:00
max   2011-12-09 12:50:00
Name: InvoiceDate, dtype: datetime64[us]

These checks describe types, typical/extreme values, distinct values, and date coverage, helping identify questions before cleaning.

In [10]:
# Missing values.
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [11]:
# Missing values percentage.
(df.isna().mean() * 100).round(2)

Invoice         0.00
StockCode       0.00
Description     0.41
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    22.77
Country         0.00
dtype: float64

**Missing values:** Customer ID is absent in **243,007** rows; Description in **4,382**. We still need to investigate these gaps.

In [12]:
# Count exact repeats across all eight columns.
df.duplicated().sum()

np.int64(34335)

**Duplicates:** **34,335 additional exact repeats** across all eight columns. The first occurrence is not flagged. Repeats need investigation before removal.

### Negative values

The summary statistics showed negative minimum values. First count them, then inspect the records to understand their meaning.

In [13]:
(df[["Quantity", "Price"]] < 0).sum()

Quantity    22950
Price           5
dtype: int64

In [14]:
df[df["Price"] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,<NA>,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,<NA>,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,<NA>,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,<NA>,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,<NA>,United Kingdom


**Negative Price:** All **5** rows describe Adjust bad debt, have Quantity 1 and lack Customer ID. They appear to be accounting adjustments.

In [15]:
df[df["Quantity"] < 0].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592,United Kingdom


The negative-quantity sample contains invoices starting with `C`. [UCI's dataset documentation](https://archive.ics.uci.edu/dataset/502/online%2Bretail%2Bii) defines this prefix as a cancellation. Next, check whether all negative-quantity rows follow this pattern.

In [16]:
negative_quantity_rows = df.loc[df["Quantity"] < 0]

negative_quantity_rows["Invoice"].str.startswith("C", na=False).value_counts()

Invoice
True     19493
False     3457
Name: count, dtype: Int64

Some negative-quantity rows have invoices without the cancellation prefix. Inspect these exceptions before drawing a conclusion.

In [17]:
negative_quantity_rows[
    ~negative_quantity_rows["Invoice"].str.startswith("C", na=False)
].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,<NA>,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,<NA>,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,<NA>,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,<NA>,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,<NA>,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,<NA>,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,<NA>,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,<NA>,United Kingdom
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,<NA>,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,<NA>,United Kingdom


In [18]:
other_negative_rows = negative_quantity_rows.loc[
    ~negative_quantity_rows["Invoice"].str.startswith("C", na=False)
]

pd.Series({
    "Total rows": len(other_negative_rows),
    "Zero price": (other_negative_rows["Price"] == 0).sum(),
    "Missing Customer ID": other_negative_rows["Customer ID"].isna().sum()
})

Total rows             3457
Zero price             3457
Missing Customer ID    3457
dtype: int64

**Negative Quantity:** **19,493** rows have cancellation-coded invoices; **3,457** do not. All 3,457 have zero Price and missing Customer ID. Sample descriptions such as 'lost' and 'damages' suggest inventory adjustments. Negative quantities are therefore not all cancellation-coded.

### Zero values

The negative-value check excludes zero. Count zero quantities and prices separately, then inspect examples.

In [19]:
(df[["Quantity", "Price"]] == 0).sum()

Quantity       0
Price       6202
dtype: int64

In [20]:
df[
    (df["Price"] == 0) & (df["Quantity"] > 0)
].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,<NA>,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,<NA>,United Kingdom
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126,United Kingdom
5904,489861,DOT,DOTCOM POSTAGE,1,2009-12-02 14:50:00,0.0,<NA>,United Kingdom
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,<NA>,United Kingdom
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,<NA>,United Kingdom
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,<NA>,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658,United Kingdom
7204,490015,21982,NaN,467,2009-12-03 12:29:00,0.0,<NA>,United Kingdom
9249,490123,84508B,NaN,184,2009-12-03 18:08:00,0.0,<NA>,United Kingdom


**Zero values:** Quantity has **0**; Price has **6,202**. The positive-quantity, zero-price sample includes missing details, named products with Customer IDs, and DOTCOM POSTAGE. `head(10)` shows examples only; it cannot establish free items versus pricing errors.

# Data Cleaning

### Duplicates

In [21]:
df = df.drop_duplicates()
df.shape

(1033036, 8)

Removed **34,335** exact repeated rows, keeping the first occurrence. **1,033,036 rows remain.**

### Missing values

In [22]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4275
Quantity            0
InvoiceDate         0
Price               0
Customer ID    235151
Country             0
dtype: int64

In [23]:
# Missing values percentage.
(df.isna().mean() * 100).round(2)

Invoice         0.00
StockCode       0.00
Description     0.41
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    22.76
Country         0.00
dtype: float64

In [24]:
(df["Description"].isna() & df["Customer ID"].isna()).sum()


np.int64(4275)

In [25]:
df.shape

(1033036, 8)

In [26]:
df = df.dropna(subset=["Customer ID"])
df.shape

(797885, 8)

In [27]:
df.isna().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

In [28]:
df.shape

(797885, 8)

Removed **235,151** rows without Customer ID because customer histories require an identity. All **4,275** missing descriptions were in that group. **797,885 rows remain, with no missing values.**

Save a copy before removing credits and Manual entries. Later cells need these records to check whether an original purchase was reversed. Keep the original row index.

In [29]:
transactions = df.copy()

### Cancellation records

In [30]:
(df[["Quantity", "Price"]] < 0).sum()

Quantity    18390
Price           0
dtype: int64

In [31]:
df[df["Quantity"] < 0]["Invoice"].str.startswith("C", na=False).value_counts()

Invoice
True    18390
Name: count, dtype: Int64

In [32]:
cancellation_rows = df[df["Invoice"].str.startswith("C", na=False)]

(cancellation_rows["Quantity"] < 0).value_counts()

Quantity
True    18390
Name: count, dtype: int64

Saved **18,390 cancellation lines** before further filtering. These are credit records, so exclude them as purchases; use the saved evidence to review their originals separately.

In [33]:
df = df[~df["Invoice"].str.startswith("C", na=False)].copy()
df.shape

(779495, 8)

Cancellation records are now outside the purchase table: **779,495 rows remain.** Removing a credit record alone does not remove its original purchase.

### Zero prices

In [34]:
(df["Price"] == 0).sum()

np.int64(70)

In [35]:
df[df["Price"] == 0].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658,United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231,United Kingdom
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108,United Kingdom
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108,United Kingdom
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.0,15070,United Kingdom
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.0,18071,United Kingdom
47126,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.0,14258,United Kingdom
48342,493899,22355,"CHARLOTTE BAG , SUKI DESIGN",10,2010-01-08 10:43:00,0.0,12417,Belgium
57619,494607,21533,RETRO SPOT LARGE MILK JUG,12,2010-01-15 12:43:00,0.0,16858,United Kingdom


In [36]:
df = df[df["Price"] > 0].copy()
df.shape

(779425, 8)

Removed **70 zero-price lines**. This project uses positive-priced order records; **779,425 rows remain.**

### Manual entries

In [37]:
df[df["StockCode"] == "M"].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
11310,490300,M,Manual,1,2009-12-04 14:19:00,0.85,12970,United Kingdom
11311,490300,M,Manual,1,2009-12-04 14:19:00,0.21,12970,United Kingdom
17386,490760,M,Manual,1,2009-12-08 09:49:00,10.00,14295,United Kingdom
17887,490881,M,Manual,1,2009-12-08 12:58:00,10.00,16210,United Kingdom
19542,490999,M,Manual,1,2009-12-08 17:26:00,15.95,13883,United Kingdom
19543,490999,M,Manual,1,2009-12-08 17:26:00,8.70,13883,United Kingdom
22960,491176,M,Manual,1,2009-12-10 11:50:00,1213.02,13091,United Kingdom
37762,492484,M,Manual,4,2009-12-17 11:43:00,0.35,15862,United Kingdom
38943,492650,M,Manual,1,2009-12-17 17:43:00,803.25,14911,EIRE
42208,492935,M,Manual,1,2009-12-21 12:18:00,74.10,13641,United Kingdom


In [38]:
df[df["StockCode"] == "M"]["Description"].value_counts()

Description
Manual    681
Name: count, dtype: int64

In [39]:
df = df[df["StockCode"] != "M"].copy()
df.shape

(778744, 8)

Removed **681 generic Manual purchase lines** because they do not identify products. **778,744 rows remain.** Manual credits and charges still matter as evidence when reconciling original purchases.

### In-depth categorical checks

In [40]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns
cat_cols

Index(['Invoice', 'StockCode', 'Description', 'Customer ID', 'Country'], dtype='str')

In [41]:
df["Country"].value_counts(dropna=False)

Country
United Kingdom          699794
Germany                  16421
EIRE                     15541
France                   13503
Netherlands               5080
Spain                     3660
Belgium                   3054
Switzerland               3005
Portugal                  2349
Australia                 1784
Channel Islands           1550
Italy                     1442
Sweden                    1314
Norway                    1284
Cyprus                    1135
Finland                   1031
Austria                    920
Denmark                    778
Greece                     657
Unspecified                518
Poland                     504
Japan                      468
USA                        408
United Arab Emirates       382
Singapore                  332
Israel                     319
Malta                      280
Canada                     228
Iceland                    222
Lithuania                  154
RSA                        122
Brazil                      94


In [42]:
# Count each invoice once to get orders per country.
country_orders = df.drop_duplicates("Invoice")["Country"].value_counts()

pd.DataFrame({
    "Orders": country_orders,
    "Percent": (country_orders / country_orders.sum() * 100).round(2)
}).head(10)


,Orders,Percent
Country,,
United Kingdom,33437,90.88
Germany,780,2.12
France,607,1.65
EIRE,543,1.48
Netherlands,223,0.61
Spain,152,0.41
Belgium,148,0.40
Sweden,102,0.28
Australia,90,0.24


In [43]:
(df["Country"] != df["Country"].str.strip()).sum()

np.int64(0)

No outer whitespace was found in Country. Keep the reported labels and rare countries; low frequency alone is not an error.

In [44]:
for col in cat_cols:
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-' * 100)

Invoice
36791
<ArrowStringArray>
['489434', '489435', '489436', '489437', '489438', '489439', '489440',
 '489441', '489442', '489443',
 ...
 '581578', '581579', '581581', '581580', '581582', '581583', '581584',
 '581585', '581586', '581587']
Length: 36791, dtype: string
----------------------------------------------------------------------------------------------------
StockCode
4630
<ArrowStringArray>
[ '85048', '79323P', '79323W',  '22041',  '21232',  '22064',  '21871',
  '21523',  '22350',  '22349',
 ...
  '23552',  '23498',  '23578',  '23580',  '23575',  '23560',  '23576',
  '23562',  '23561',  '23843']
Length: 4630, dtype: string
----------------------------------------------------------------------------------------------------
Description
5282
['15CM CHRISTMAS GLASS BALL 20 LIGHTS' 'PINK CHERRY LIGHTS'
 ' WHITE CHERRY LIGHTS' ... 'SILVER AND BLACK ORBIT NECKLACE'
 'CREAM HANGING HEART T-LIGHT HOLDER' 'PAPER CRAFT , LITTLE BIRDIE']
------------------------------------------------

In [45]:
(df["Description"] != df["Description"].str.strip()).sum()

np.int64(158242)

In [46]:
df["Description"] = df["Description"].str.strip()
df["Description"].nunique()

5240

Trimmed outer whitespace in Description. Use StockCode for grouping products; do not merge products merely because their descriptions sound similar.

In [47]:
df[df["StockCode"].isin([
    "POST", "C2", "BANK CHARGES", "TEST001", "TEST002",
    "PADS", "ADJUST", "D", "ADJUST2", "SP1002", "DOT"
])]["StockCode"].value_counts()

StockCode
POST            1803
C2               248
ADJUST            32
BANK CHARGES      31
PADS              17
DOT               16
TEST001            9
D                  5
ADJUST2            3
SP1002             2
TEST002            1
Name: count, dtype: int64[pyarrow]

In [48]:
df[df["StockCode"].isin([
    "POST", "C2", "BANK CHARGES", "TEST001", "TEST002",
    "PADS", "ADJUST", "D", "ADJUST2", "SP1002", "DOT"
])][["StockCode", "Description"]].drop_duplicates()

,StockCode,Description
89,POST,POSTAGE
9292,C2,CARRIAGE
18466,BANK CHARGES,Bank Charges
27994,TEST001,This is a test product.
39411,TEST002,This is a test product.
62299,PADS,PADS TO MATCH ALL CUSHIONS
70976,ADJUST,Adjustment by john on 26/01/2010 16
71058,ADJUST,Adjustment by john on 26/01/2010 17
160443,D,Discount
249672,ADJUST2,Adjustment by Peter on Jun 25 2010


In [49]:
df = df[~df["StockCode"].isin([
    "POST", "C2", "BANK CHARGES", "TEST001", "TEST002",
    "ADJUST", "D", "ADJUST2", "DOT"
])].copy()

df.shape

(776596, 8)

Removed **2,148** postage, fee, discount, adjustment and test lines. Kept named products such as PADS and SP1002. **776,596 positive product rows remain.**

In [50]:
# Product lines per invoice: the ten largest orders.
df["Invoice"].value_counts().head(10)

Invoice
576339    541
579196    532
580727    528
578270    441
573576    434
567656    420
567183    391
575607    376
571441    363
570488    352
Name: count, dtype: int64[pyarrow]

In [51]:
(df.groupby("Invoice")["Customer ID"].nunique() > 1).sum()

np.int64(0)

Each retained invoice belongs to one customer and can contain many product lines. Repeated invoice numbers are normal; they are not duplicate rows by themselves.

In [52]:
df[~df["Customer ID"].str.fullmatch(r"\d{5}", na=False)]["Customer ID"].unique()

<ArrowStringArray>
[]
Length: 0, dtype: string

**Identifiers:** Keep Invoice, StockCode and Customer ID as strings for grouping. The separate audit found no decimal suffixes, mixed StockCode types, outer whitespace or case-only StockCode variants. No conversion was needed.

In [53]:
# Product lines per customer: the ten most active customers.
df["Customer ID"].value_counts().head(10)

Customer ID
17841    12425
14911    10909
12748     6557
14606     6347
14096     5095
15311     4303
14156     3994
14646     3819
13089     3315
16549     3093
Name: count, dtype: int64[pyarrow]

These frequencies count **product-line rows**, not separate orders or units. Later, use distinct invoice counts when building purchase-frequency features.

### In-depth numerical checks

In [54]:
df[["Quantity", "Price"]].describe()

,Quantity,Price
count,776596.000000,776596.000000
mean,13.518940,2.934497
std,146.094921,4.293580
min,1.000000,0.001000
25%,2.000000,1.250000
50%,6.000000,1.950000
75%,12.000000,3.750000
max,80995.000000,649.500000


Quantity and Price have large upper extremes. Review the actual transactions before deciding whether an unusual value needs cleaning.

In [55]:
df.sort_values("Quantity", ascending=False).head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346,United Kingdom
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902,Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902,Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902,Denmark
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902,Denmark
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom
135030,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom


Check the saved cancellation records for the two largest purchase quantities.

In [56]:
cancellation_rows[
    (cancellation_rows["Customer ID"] == "16446")
    & (cancellation_rows["StockCode"] == "23843")
]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446,United Kingdom


In [57]:
cancellation_rows[
    (cancellation_rows["Customer ID"] == "12346")
    & (cancellation_rows["StockCode"] == "23166")
]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346,United Kingdom


The **80,995** and **74,215** quantities have exact full credits **12** and **16 minutes** later. The final same-day rule below handles both originals.

### Same-day purchase reversals

Removing the credit lines is not enough: when a customer cancels an order **on the same day**, the original purchase never really happened, so it should not count as a purchase either.

The steps below find those originals:
1. Prepare purchases and credits with their money amounts.
2. Check the reviewed Manual corrections.
3. Match whole invoices that were fully credited the same day.
4. Match remaining single product lines that were credited the same day.
5. Remove the matched original purchases.

Credits on a later day, or partial credits, do not remove the original purchase.

#### 1. Prepare the transactions

Use the saved copy that still contains the credits and Manual entries. `amount` is Price × Quantity, rounded to pennies so money values can be compared exactly.

In [58]:
matching_data = transactions.copy()
matching_data["source_row"] = matching_data.index
matching_data["day"] = matching_data["InvoiceDate"].dt.normalize()
matching_data["amount"] = (matching_data["Price"] * matching_data["Quantity"]).round(2)

In [59]:
cancellation_mask = matching_data["Invoice"].str.upper().str.startswith("C", na=False)

purchases = matching_data[
    ~cancellation_mask
    & (matching_data["Quantity"] > 0)
    & (matching_data["Price"] > 0)
].copy()

credits = matching_data[
    cancellation_mask
    & (matching_data["Quantity"] < 0)
    & (matching_data["Price"] > 0)
].copy()

In [60]:
non_product_codes = [
    "M", "POST", "C2", "BANK CHARGES", "TEST001", "TEST002",
    "ADJUST", "D", "ADJUST2", "DOT"
]

#### 2. Check the Manual corrections

During the review, eleven invoices were found whose same-day credit included a generic Manual (`M`) line. For each one, add up the customer's Manual entries after the purchase on that day:

- **Net £0:** the Manual credit was cancelled by a matching Manual charge (a rebill), so **keep** the purchase.
- **Net = minus the purchase amount:** the purchase was fully credited, so **exclude** it.

In [61]:
# Original invoice -> its same-day credit invoice, found during the review.
reviewed_manual_links = {
    "495232": "C495234", "497337": "C497338", "497947": "C497951",
    "506731": "C506739", "515123": "C515125", "519735": "C519737",
    "521038": "C521041", "542303": "C542348", "544574": "C544577",
    "545845": "C545891", "556444": "C556445"
}

keep_rebilled_invoices = []
neutral_credit_ids = []
manual_checks = []

In [62]:
for invoice, credit_invoice in reviewed_manual_links.items():
    sale = purchases[purchases["Invoice"] == invoice]
    customer = sale["Customer ID"].iloc[0]
    day = sale["day"].iloc[0]
    original_amount = round(sale["amount"].sum(), 2)

    # The customer's Manual entries later on the same day.
    manual_entries = matching_data[
        (matching_data["Customer ID"] == customer)
        & (matching_data["day"] == day)
        & (matching_data["StockCode"] == "M")
        & (matching_data["InvoiceDate"] > sale["InvoiceDate"].max())
    ]
    manual_net = round(manual_entries["amount"].sum(), 2)

    if manual_net == 0:
        decision = "Keep: offsetting charge"
        keep_rebilled_invoices.append(invoice)
        neutral_credit_ids.append(credit_invoice)
    else:
        decision = "Exclude: full net credit"

    manual_checks.append([invoice, day, original_amount, manual_net, decision])

In [63]:
pd.DataFrame(
    manual_checks,
    columns=["Invoice", "Day", "Original amount", "Manual net", "Decision"]
)

,Invoice,Day,Original amount,Manual net,Decision
0,495232,2010-01-22,1193.89,0.00,Keep: offsetting charge
1,497337,2010-02-08,365.10,0.00,Keep: offsetting charge
2,497947,2010-02-15,1508.65,0.00,Keep: offsetting charge
3,506731,2010-05-04,269.34,0.00,Keep: offsetting charge
4,515123,2010-07-08,715.02,-715.02,Exclude: full net credit
5,519735,2010-08-20,152.40,-152.40,Exclude: full net credit
6,521038,2010-09-02,1055.75,0.00,Keep: offsetting charge
7,542303,2011-01-27,1715.85,-1715.85,Exclude: full net credit
8,544574,2011-02-21,320.69,0.00,Keep: offsetting charge
9,545845,2011-03-07,16.05,-16.05,Exclude: full net credit


#### 3. Match whole invoices

Calculate each invoice's total amount and time range with `groupby`. Keep the list of its row numbers so its product lines can be selected later. A credit invoice matches a purchase invoice when the customer, the day and the total amount are the same.

In [64]:
purchase_invoices = purchases.groupby(["Customer ID", "Invoice"]).agg(
    first=("InvoiceDate", "min"),
    last=("InvoiceDate", "max"),
    amount=("amount", "sum"),
    rows=("source_row", list),
    days=("day", "nunique")
).reset_index()

purchase_invoices["amount"] = purchase_invoices["amount"].round(2)
purchase_invoices["day"] = purchase_invoices["first"].dt.normalize()

In [65]:
credit_invoices = credits.groupby(["Customer ID", "Invoice"]).agg(
    first=("InvoiceDate", "min"),
    last=("InvoiceDate", "max"),
    amount=("amount", "sum"),
    rows=("source_row", list),
    days=("day", "nunique")
).reset_index()

# Credits are negative; make them positive to compare with purchases.
credit_invoices["amount"] = -credit_invoices["amount"].round(2)
credit_invoices["day"] = credit_invoices["first"].dt.normalize()

In [66]:
# Join only the same customer, calendar day and full amount.
invoice_candidates = credit_invoices.merge(
    purchase_invoices, on=["Customer ID", "day", "amount"],
    suffixes=("_credit", "_purchase")
)

# The complete purchase must precede the credit.
invoice_candidates = invoice_candidates[
    (invoice_candidates["last_purchase"] < invoice_candidates["first_credit"])
    & (invoice_candidates["days_purchase"] == 1)
    & (invoice_candidates["days_credit"] == 1)
].copy()

Check the products too: the credited product and price must occur on the original, and the credited quantity must not exceed the purchased quantity. Manual credits affect the amount only.

In [67]:
invoice_candidates["products_fit"] = False

# iterrows lets us check one possible invoice pair at a time.
for index, match in invoice_candidates.iterrows():
    original = matching_data.loc[match["rows_purchase"]]
    credit = matching_data.loc[match["rows_credit"]]
    credit = credit[credit["StockCode"] != "M"]

    bought = original.groupby(["StockCode", "Price"])["Quantity"].sum()
    refunded = -credit.groupby(["StockCode", "Price"])["Quantity"].sum()

    # Align the same products/prices; a missing product has zero available units.
    available = bought.reindex(refunded.index, fill_value=0)
    invoice_candidates.loc[index, "products_fit"] = (refunded <= available).all()

invoice_candidates = invoice_candidates[invoice_candidates["products_fit"]]

Go through the credits in time order and accept one matching original per credit. The list `used_invoices` makes sure the same original is never used twice. The rebilled invoices from step 2 are skipped.

In [68]:
used_invoices = []
invoice_matches = []

for credit_invoice in credit_invoices.sort_values(["first", "Invoice"])["Invoice"]:
    if credit_invoice in neutral_credit_ids:
        continue

    available = invoice_candidates[
        (invoice_candidates["Invoice_credit"] == credit_invoice)
        & ~invoice_candidates["Invoice_purchase"].isin(used_invoices)
    ]

    if len(available) == 1:
        match = available.iloc[0]
        if match["Invoice_purchase"] not in keep_rebilled_invoices:
            invoice_matches.append(match)
            used_invoices.append(match["Invoice_purchase"])

invoice_matches = pd.DataFrame(invoice_matches)

In [69]:
invoice_matches[["Invoice_purchase", "Invoice_credit", "amount"]].head()

,Invoice_purchase,Invoice_credit,amount
164,492650,C492651,803.25
241,493189,C493190,67.49
242,493188,C493191,20.41
165,493897,C493898,780.52
128,494348,C494352,546.95


Collect the original product rows from these matches. Save the used purchase and credit indices in lists; extend adds several indices to a list. No rows have been removed from df yet.

In [70]:
used_purchase_rows = []
used_credit_rows = []
exclusion_records = []

for index, match in invoice_matches.iterrows():
    used_purchase_rows.extend(match["rows_purchase"])
    used_credit_rows.extend(match["rows_credit"])

    # Only product rows that are still in df are removed.
    for row_index in df[df.index.isin(match["rows_purchase"])].index:
        exclusion_records.append([row_index, match["Invoice_purchase"], match["Invoice_credit"], "Whole invoice credited the same day"])

#### 4. Match single product lines

For the remaining credits, match the same customer, product, price, quantity and day, with the credit after the purchase. If several identical purchases fit, use the latest one. Each line is used only once.

In [71]:
remaining_purchases = purchases[
    purchases.index.isin(df.index)
    & ~purchases.index.isin(used_purchase_rows)
]

remaining_credits = credits[
    ~credits.index.isin(used_credit_rows)
    & ~credits["StockCode"].isin(non_product_codes)
].copy()

# Positive quantities let us join bought 50 with credited -50.
remaining_credits["Quantity"] = -remaining_credits["Quantity"]

In [72]:
item_candidates = remaining_credits.merge(
    remaining_purchases,
    on=["Customer ID", "StockCode", "Price", "Quantity", "day"],
    suffixes=("_credit", "_purchase")
)

item_candidates = item_candidates[
    item_candidates["InvoiceDate_purchase"] < item_candidates["InvoiceDate_credit"]
]

Sort credits from earliest to latest. For each credit, put the latest eligible purchase first; original row indices break ties.

In [73]:
item_candidates = item_candidates.sort_values(
    ["InvoiceDate_credit", "source_row_credit", "InvoiceDate_purchase", "source_row_purchase"],
    ascending=[True, True, False, False]
)

In [74]:
matched_purchase_rows = []
matched_credit_rows = []

for index, match in item_candidates.iterrows():
    purchase_index = int(match["source_row_purchase"])
    credit_index = int(match["source_row_credit"])

    # Skip lines that were already used by an earlier match.
    if purchase_index in matched_purchase_rows or credit_index in matched_credit_rows:
        continue

    matched_purchase_rows.append(purchase_index)
    matched_credit_rows.append(credit_index)
    exclusion_records.append([purchase_index, match["Invoice_purchase"], match["Invoice_credit"], "Single product line credited the same day"])

#### 5. Review and remove the matched purchases

Each row below is one original product line to remove. Check the counts before dropping anything.

In [75]:
same_day_exclusions = pd.DataFrame(
    exclusion_records,
    columns=["source_row_purchase", "original_invoice", "credit_invoice", "reason"]
)

same_day_exclusions["reason"].value_counts()

reason
Whole invoice credited the same day          918
Single product line credited the same day    384
Name: count, dtype: int64

In [76]:
same_day_exclusions[["original_invoice", "credit_invoice", "reason"]].head()

,original_invoice,credit_invoice,reason
0,493189,C493190,Whole invoice credited the same day
1,493189,C493190,Whole invoice credited the same day
2,493189,C493190,Whole invoice credited the same day
3,493189,C493190,Whole invoice credited the same day
4,493189,C493190,Whole invoice credited the same day


In [77]:
# Confirm each selected original appears once and is still in df.
print("Selected rows:", len(same_day_exclusions))
print("Repeated selections:", same_day_exclusions["source_row_purchase"].duplicated().sum())
print("All selected rows exist:", same_day_exclusions["source_row_purchase"].isin(df.index).all())

Selected rows: 1302
Repeated selections: 0
All selected rows exist: True


select `drop_indices`, then `df.drop`.

In [78]:
# Select the row indices identified by the reviewed cancellation rules.
drop_indices = df[
    df.index.isin(same_day_exclusions["source_row_purchase"])
].index

print("Original purchase rows removed:", len(drop_indices))
df.drop(drop_indices, axis=0, inplace=True)
df.shape


Original purchase rows removed: 1302


(775294, 8)

**Result:** Removed **1,302 original purchase rows**, separate from the cancellation records excluded earlier. **775,294 rows remain.** The **218 product rows** from the six rebilled invoices are kept.

In [79]:
df.sort_values("Quantity", ascending=False).head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902,Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902,Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902,Denmark
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902,Denmark
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom
135030,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom
93677,498152,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,9456,2010-02-17 10:51:00,0.30,13902,Denmark


Remaining large quantities are consistent with bulk orders. No additional rows are removed solely because Quantity is large.

In [80]:
df.sort_values("Price", ascending=False).head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
748143,556446,22502,PICNIC BASKET WICKER 60 PIECES,1,2011-06-10 15:33:00,649.5,15098,United Kingdom
292603,517909,22655,VINTAGE RED KITCHEN CABINET,1,2010-08-03 10:26:00,295.0,18260,United Kingdom
292666,517930,22655,VINTAGE RED KITCHEN CABINET,1,2010-08-03 10:38:00,295.0,15727,United Kingdom
292665,517930,22656,VINTAGE BLUE KITCHEN CABINET,1,2010-08-03 10:38:00,295.0,15727,United Kingdom
577097,540647,22655,VINTAGE RED KITCHEN CABINET,1,2011-01-10 14:57:00,295.0,17406,United Kingdom
290300,517650,22655,VINTAGE RED KITCHEN CABINET,1,2010-07-30 14:00:00,295.0,13015,United Kingdom
403382,528051,22656,VINTAGE BLUE KITCHEN CABINET,1,2010-10-20 11:41:00,295.0,15874,United Kingdom
287812,517393,22655,VINTAGE RED KITCHEN CABINET,2,2010-07-29 10:00:00,295.0,13340,United Kingdom
507927,536835,22655,VINTAGE RED KITCHEN CABINET,1,2010-12-02 18:06:00,295.0,13145,United Kingdom
466662,533370,22656,VINTAGE BLUE KITCHEN CABINET,2,2010-11-17 10:31:00,295.0,15874,United Kingdom


In [81]:
cancellation_rows[
    cancellation_rows["Customer ID"] == "15098"
].sort_values("InvoiceDate")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
748142,C556445,M,Manual,-1,2011-06-10 15:31:00,38970.00,15098,United Kingdom
748153,C556448,22502,PICNIC BASKET WICKER SMALL,-60,2011-06-10 15:39:00,4.95,15098,United Kingdom


**Basket audit:** The Manual credit of **38,970** supports excluding the 60-unit original; the later one-unit order **556446** stays. This lookup uses preserved credit evidence and does not apply another removal.

In [82]:
import plotly.express as px

# A random sample of 20,000 rows keeps the notebook small; the shape of the distribution is the same.
df_sample = df.sample(20000, random_state=42)

for col in ["Quantity", "Price"]:
    px.histogram(data_frame=df_sample, x=col, title=col).show()
    px.box(data_frame=df_sample, x=col, title=col).show()

Histograms and box plots describe the distributions. Statistical outliers are not automatic errors; no quantity- or price-based clipping has been applied.

### Final cleaning check

Check for missing values, repeated records and invalid purchase entries before feature engineering. All counts should be zero; the final shape should be **(775294, 8)**.

In [83]:
print("Missing values:", df.isna().sum().sum())
print("Exact duplicates:", df.duplicated().sum())
print("Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("Price <= 0:", (df["Price"] <= 0).sum())
print("Manual rows:", (df["StockCode"] == "M").sum())
print("Cancellation rows:", df["Invoice"].str.startswith("C", na=False).sum())
df.shape


Missing values: 0


Exact duplicates: 0
Quantity <= 0: 0
Price <= 0: 0
Manual rows: 0
Cancellation rows: 0


(775294, 8)

In [84]:
# The .gz ending compresses the file; pd.read_csv can read it directly.
df.to_csv("data/cleaned_purchases.csv.gz", index=False)

**Result:** 775,294 clean purchase lines from 5,851 customers are saved in `data/cleaned_purchases.csv.gz`. Continue with `2_Feature_Engineering_and_EDA.ipynb`.